# CatBoost Pipeline V1
Production-ready starter notebook based on your Logistic Regression pipeline.

In [10]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, balanced_accuracy_score
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from joblib import dump


In [ ]:
class DateFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self,date_column="Date"):
        self.date_column=date_column
        
    def fit(self,X,y=None):
        d=pd.to_datetime(X[self.date_column],errors="coerce")
        self.median_date_=d.dropna().median()
        return self
    
    def transform(self,X):
        X=X.copy()
        X[self.date_column]=pd.to_datetime(X[self.date_column],errors="coerce")

        X["DateMissing"]=X[self.date_column].isna().astype(int)
        
        X[self.date_column]=X[self.date_column].fillna(self.median_date_)
        # X["Date_Year"]=X[self.date_column].dt.year
        
        # Cast as string categories so CatBoost treats them natively as categoricals
        X["Date_Month"] = X[self.date_column].dt.month.astype(str)
        X["Date_DayOfWeek"] = X[self.date_column].dt.dayofweek.astype(str)
        X["Date_DayOfMonth"] = X[self.date_column].dt.day.astype(str)
        
        X.drop(columns=[self.date_column],inplace=True)
        return X

class NumericFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns if columns else ["Value", "No_Wall_Types"]
        
    def fit(self, X, y=None):
        self.medians_ = {}
        for col in self.columns:
            if col in X.columns:
                self.medians_[col] = X[col].median()
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Impute all numeric columns
        for col in self.columns:
            if col in X.columns:
                X[col] = X[col].fillna(self.medians_[col])
        
        # Log transform only Value (skewed)
        if "Value" in X.columns:
            X["Value"] = np.log1p(X["Value"])
            
        return X

class EstimatorFeatureTransformer(BaseEstimator,TransformerMixin):
    def __init__(self,column="Priced_By"):
        self.column=column

    def fit(self,X,y=None): 
        return self
    
    def transform(self,X):
        X=X.copy()
        s=X[self.column].fillna("MISSING").astype(str)
        
        X["EstimatorCount"]=s.str.split("/").apply(len)
        
        X.loc[s.str.upper().eq("MISSING"),"EstimatorCount"]=0
        
        X["EstimatorCount"]=X["EstimatorCount"].astype(int)
        
        X["EstimatorMissing"]=s.str.upper().eq("MISSING").astype(int)
        return X

# Safe categorical filler to prevent NaN float crashes in CatBoost
class CategoricalImputer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                X[col] = X[col].fillna("MISSING").astype(str)
        return X

In [ ]:
# Load your dataframe into quotation_data_df before running

quotation_data_path = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Quotation Data.xlsx"
# quotation_data_df = pd.read_excel(quotation_data_path, dtype=dtype_mapping)
quotation_data_df = pd.read_excel(quotation_data_path)

raw_features=[
"Date","Value","No_Wall_Types","Priced_By","Suburb","Client_Clean",
"Timber_RW","RC_Pile","Steel_Beam","Sheetpile","Anchor","Block",
"Shotcrete","Capping_Beam","Earthwork","Concrete_Slab","Precast",
"Culvert","Slip_Repair","Soil_Nail","Rock_RW","Bridge","Concrete",
"Design_and_Build","Budget","Drill_Only","Labour_Only",
"Driven_Pile","Palisade","Boardwalk","Soldier","Insitu",
"Barrier","Noise_RW","Base","Casing","Crib",
"DayWork","Flood_Repair","Micro_Pile","Reno","Temp_RW","Other"]

target="Success"

quotation_data_df["Date"]=pd.to_datetime(quotation_data_df["Date"],errors="coerce")
quotation_data_df=quotation_data_df.sort_values("Date").reset_index(drop=True)

X=quotation_data_df[raw_features].copy()
y=quotation_data_df[target]

split=int(len(X)*0.8)
X_train,X_test=X.iloc[:split],X.iloc[split:]
y_train,y_test=y.iloc[:split],y.iloc[split:]

# Define which columns will be categorical after transformation
# These are the columns that will exist AFTER all transformers
CATEGORICAL_COLUMNS = [
    "Priced_By",
    "Suburb", 
    "Client_Clean",
    "Date_Month",
    "Date_DayOfWeek",
    "Date_DayOfMonth"
]

model = CatBoostClassifier(
    cat_features=CATEGORICAL_COLUMNS,
    verbose=0,
    random_seed=42
    )

# Pipeline encapsulates engineering steps to completely stop data leakage
catboos_pipeline = Pipeline([
    ("date", DateFeatureTransformer()),
    ("estimator", EstimatorFeatureTransformer(column="Priced_By")),
    ("numeric", NumericFeatureTransformer(columns=["Value", "No_Wall_Types"])),
    ("cat_imputer", CategoricalImputer(columns=["Priced_By","Suburb", "Client_Clean"])),
    ("classifier", model)
])

param_dist = {
    "classifier__depth": [4, 6, 8],
    "classifier__iterations": [300, 500, 800],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__l2_leaf_reg": [1, 3, 5, 7, 10],
    "classifier__random_strength": [0, 1, 2, 5],
    "classifier__auto_class_weights": ["Balanced", "SqrtBalanced"]
}

tscv=TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    catboos_pipeline,
    param_dist,
    n_iter=30,
    cv=tscv,
    scoring="balanced_accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=2)

search.fit(X_train,y_train)

best=search.best_estimator_
print("\n" + "="*60)
print("Best Parameters:")
print(search.best_params_)
print(f"Best CV Balanced Accuracy: {search.best_score_:.4f}")
print("="*60)

# Evaluation
# ================================================================
proba=best.predict_proba(X_test)[:,1]
threshold=0.50
pred=(proba>=threshold).astype(int)

print(classification_report(y_test,pred))
print("AUC:",roc_auc_score(y_test,proba))

# Feature Importance
# ================================================================
importance = pd.DataFrame({
    "Feature": best.named_steps['classifier'].feature_names_,
    "Importance": best.named_steps['classifier'].get_feature_importance()
}).sort_values("Importance", ascending=False)

print("\n" + "="*60)
print("Top 20 Feature Importances:")
print(importance.head(20))
print("="*60)

# Save Model
# ================================================================
dump(best,"CatBoost_V1.joblib")


Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best Parameters:
{'classifier__random_strength': 1, 'classifier__learning_rate': 0.01, 'classifier__l2_leaf_reg': 15, 'classifier__iterations': 1000, 'classifier__depth': 8, 'classifier__bagging_temperature': 2, 'classifier__auto_class_weights': 'SqrtBalanced'}
Best CV Balanced Accuracy: 0.6865
              precision    recall  f1-score   support

           0       0.86      0.72      0.78       749
           1       0.29      0.49      0.36       175

    accuracy                           0.68       924
   macro avg       0.57      0.60      0.57       924
weighted avg       0.75      0.68      0.70       924

AUC: 0.6946099561319855

Top 20 Feature Importances:
             Feature  Importance
2          Priced_By   20.122360
4       Client_Clean   16.996859
45   Date_DayOfMonth   11.008376
3             Suburb   10.128019
43        Date_Month    9.872645
0              Value    8.842130
44    Date_DayOfWeek    7.890

['CatBoost_V1.joblib']

In [15]:
# Predict new Excel
import pandas as pd
from joblib import load

# --------------------------------------------------
# IMPORTANT:
# Import or define your custom transformers first
# --------------------------------------------------
# from transformers import *

# Load model
model = load("CatBoost_V1.joblib")

# Load new quotations
new_df = pd.read_excel(r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Quotation Data Test.xlsx")


raw_features = [
    "Date","Value","No_Wall_Types","Priced_By","Suburb","Client_Clean",
    "Timber_RW","RC_Pile","Steel_Beam","Sheetpile","Anchor","Block",
    "Shotcrete","Capping_Beam","Earthwork","Concrete_Slab","Precast",
    "Culvert","Slip_Repair","Soil_Nail","Rock_RW","Bridge","Concrete",
    "Design_and_Build","Budget","Drill_Only","Labour_Only",
    "Driven_Pile","Palisade","Boardwalk","Soldier","Insitu",
    "Barrier","Noise_RW","Base","Casing","Crib",
    "DayWork","Flood_Repair","Micro_Pile","Reno","Temp_RW","Other"
]

X_new = new_df[raw_features].copy()

# Predict
proba = model.predict_proba(X_new)[:,1]

threshold = 0.40

prediction = (proba >= threshold).astype(int)

new_df["Success_Probability"] = proba
new_df["Prediction"] = prediction
new_df["Prediction_Label"] = new_df["Prediction"].map({
    1: "Profit",
    0: "Loss"
})

output_file = r"C:\Users\Phong\Downloads\Quotation Prediction Cat.xlsx"

new_df.to_excel(output_file, index=False)

print(f"Saved to:\n{output_file}")


Saved to:
C:\Users\Phong\Downloads\Quotation Prediction Cat.xlsx
